# Theme 3: Agent Comparison & Temporal Benchmarking

**RQ6:** Which AI agent is *best at bug fixing*?  
**RQ8:** How does performance compare *before and after the AIDev cutoff*?

Dataset: `mabujadallah/GitHub-Agentic-PR-Dataset`  
Coverage: Dec 2024 – Feb 2026 (15 months)  
AIDev period: Dec 2024 – Jul 2025 | Post-AIDev: Aug 2025 – Feb 2026

**Methodology notes:**
- **Effect sizes + adjusted p-values.** With N in the thousands, raw p-values are nearly always significant. Every test reports an **odds ratio with 95% CI** (merge-rate / proportion tests) or **Cliff's delta** (continuous tests), and p-values are **Benjamini-Hochberg** adjusted across each family of tests. A result that is significant but has OR≈1 or a negligible delta is flagged as practically negligible.
- **RQ6 matched baseline.** RQ6 is reported against both the unmatched human baseline and a repo+time-matched one, so "best agent" is not an artefact of repo mix.
- **RQ8 fixed repo set.** Because the dataset's repo coverage shifts over time, the temporal (AIDev vs post-AIDev) comparison is run on the **AIDev repo set only** (`restrict_to_aidev_repos=True`) — otherwise a period difference could just be a change in which repos are present.

> **v2 — AIDev repository sample.** This notebook re-runs the analysis restricted to the **true AIDev repository set** (1,743 repos from `hao-li/AIDev`, excl. Codex; 1,615 present in our data) via `load_fix_prs(restrict_to_repos=AIDEV_REPOS_TRUE)`. Figures are written to the `*_figures_v2/` folders. The broad-collection results (v1) are unchanged.

In [1]:
%pip install matplotlib seaborn scipy pyarrow fsspec requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\Mahmoudabujadallah\CLI\.venv\Scripts\python.exe -m pip install --upgrade pip


In [2]:
import sys
sys.path.insert(0, '.')
from analysis_utils import (
    load_fix_prs, load_commits, load_commit_details, build_revision_stats,
    merge_rate, chi_square, mann_whitney, sig_label,
    odds_ratio_ci, cliffs_delta, bh_correct,
    build_matched_human_baseline,
    set_plot_style, save_fig,
    AGENTS, AGENT_COLORS, THEME1_DIR, THEME2_DIR, THEME3_DIR,
    THEME1_DIR_V2, THEME2_DIR_V2, THEME3_DIR_V2, load_aidev_repo_set,
    MIN_N_PROP, MIN_N_MEDIAN,
)
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline
set_plot_style()

In [3]:
AIDEV_REPOS_TRUE = load_aidev_repo_set()
# ── Load data ────────────────────────────────────────────────────────
df         = load_fix_prs(restrict_to_repos=AIDEV_REPOS_TRUE)
agents_df  = df[df['is_agent'] & df['agent'].isin(AGENTS)].copy()
human_df   = df[~df['is_agent']].copy()

# Repo- and time-matched human baseline for RQ6
matched    = build_matched_human_baseline(df)
m_human_df = matched[~matched['is_agent']].copy()

commits    = load_commits()
details    = load_commit_details()
rev_stats  = build_revision_stats(df, commits, details)
agent_rev  = rev_stats[rev_stats['agent'].isin(AGENTS)].copy()
human_rev  = rev_stats[~rev_stats['is_agent']].copy()
print('All data loaded.')

  AIDev repo set loaded: 1,743 repos
Loading fix PRs from HuggingFace ...


  AIDev repo coverage: 8,959 distinct repos
  Survivorship cutoff at 2026-01-29: dropped 33,123 recent PRs


  Restricted to 1,743 given repos: kept 305,739 of 371,577 PRs across 1,585 repos
  Fix PRs loaded: 305,739  |  Agent: 42,284  |  Human: 263,455


  Matched baseline: 42,284 agent PRs + 52,387 matched human PRs (from 263,455 human PRs)
Loading commits from HuggingFace ...


  Commits loaded: 1,156,238
Loading commit details from HuggingFace ...


  Commit details loaded: 7,451,150


All data loaded.


## RQ6: Which AI agent is best at bug fixing?

Metrics: merge rate, time to merge, revision rate, revision effort.  
Statistical tests vs Human baseline: chi-square (merge rate), Mann-Whitney U (continuous).

In [4]:
# Overall merge rate — each agent vs unmatched and repo+time-matched human baselines.
# Odds ratio with 95% CI; BH-adjusted p across the four agents (per baseline family).
hu_m, hu_t, hu_r = merge_rate(human_df)
hm_m, hm_t, hm_r = merge_rate(m_human_df)

pv_u, pv_m, rows = [], [], []
for agent in AGENTS:
    a_m, a_t, a_r = merge_rate(agents_df[agents_df['agent'] == agent])
    or_u, lo_u, hi_u = odds_ratio_ci(a_m, a_t, hu_m, hu_t)
    or_m, lo_m, hi_m = odds_ratio_ci(a_m, a_t, hm_m, hm_t)
    _, p_u = chi_square(a_m, a_t, hu_m, hu_t); pv_u.append(p_u)
    _, p_m = chi_square(a_m, a_t, hm_m, hm_t); pv_m.append(p_m)
    rows.append((agent, a_m, a_t, a_r, or_u, lo_u, hi_u, or_m, lo_m, hi_m))
pa_u, pa_m = bh_correct(pv_u), bh_correct(pv_m)

print(f"{'Group':<13}{'Merged':>8}{'Total':>9}{'Rate%':>7}   "
      f"{'OR vs unmatched':<26}{'OR vs matched':<26}")
print('-' * 92)
print(f"{'Human (unm)':<13}{hu_m:>8,}{hu_t:>9,}{hu_r:>6.1f}%")
print(f"{'Human (mat)':<13}{hm_m:>8,}{hm_t:>9,}{hm_r:>6.1f}%")
for (agent, a_m, a_t, a_r, or_u, lo_u, hi_u, or_m, lo_m, hi_m), pu, pm in zip(rows, pa_u, pa_m):
    s_u = f"{or_u:.2f} [{lo_u:.2f},{hi_u:.2f}] {sig_label(pu)}"
    s_m = f"{or_m:.2f} [{lo_m:.2f},{hi_m:.2f}] {sig_label(pm)}"
    print(f"{agent:<13}{a_m:>8,}{a_t:>9,}{a_r:>6.1f}%   {s_u:<26}{s_m:<26}")
print("\nBH-adjusted p; OR>1 => agent merges more often than that human baseline.")

Group          Merged    Total  Rate%   OR vs unmatched           OR vs matched             
--------------------------------------------------------------------------------------------
Human (unm)   224,937  263,455  85.4%
Human (mat)    45,198   52,387  86.3%
Copilot         4,214    6,764  62.3%   0.28 [0.27,0.30] ***      0.26 [0.25,0.28] ***      
Cursor         16,456   18,367  89.6%   1.47 [1.40,1.55] ***      1.37 [1.30,1.44] ***      
Claude_Code    12,995   15,176  85.6%   1.02 [0.97,1.07] ns       0.95 [0.90,1.00] *        
Devin             924    1,977  46.7%   0.15 [0.14,0.16] ***      0.14 [0.13,0.15] ***      

BH-adjusted p; OR>1 => agent merges more often than that human baseline.


In [5]:
# Figure: merge rate bar chart
groups = AGENTS + ['Human']
rates  = []
for g in groups:
    if g == 'Human':
        _, _, r = merge_rate(human_df)
    else:
        _, _, r = merge_rate(agents_df[agents_df['agent'] == g])
    rates.append(r)

colors = [AGENT_COLORS[g] for g in groups]
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(groups, rates, color=colors, edgecolor='white', linewidth=0.5)
for bar, val in zip(bars, rates):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', fontsize=10)
ax.set_ylabel('Merge Rate (%)')
ax.set_ylim(0, 105)
ax.set_title('RQ6: Bug-Fix Merge Rate by Agent vs Human')
fig.tight_layout()
save_fig(fig, 'rq6_merge_rate', THEME3_DIR_V2)

  -> Saved: results\theme3_figures_v2\rq6_merge_rate.png


WindowsPath('results/theme3_figures_v2/rq6_merge_rate.png')

In [6]:
# Time to merge — median + Mann-Whitney vs Human, with Cliff's delta effect size.
h_ttm = human_df.loc[human_df['is_merged'], 'hours_to_merge']
pvals, rows = [], []
for agent in AGENTS:
    sub = agents_df[(agents_df['agent'] == agent) & agents_df['is_merged']]['hours_to_merge']
    u, p = mann_whitney(sub, h_ttm)
    delta, mag = cliffs_delta(sub, h_ttm)
    pvals.append(p)
    rows.append((agent, sub.median(), delta, mag))
padj = bh_correct(pvals)

print("{:<14}{:>14}   {:<28}".format('Group', 'Median TTM(h)', 'Cliff delta vs Human'))
print('-' * 60)
print("{:<14}{:>14.2f}".format('Human', h_ttm.median()))
for (agent, med, delta, mag), p in zip(rows, padj):
    print("{:<14}{:>14.2f}   delta={:+.3f} ({}) {}".format(
        agent, med, delta, mag, sig_label(p)))
print("\nBH-adjusted p; delta<0 => agent merges faster than Human. "
      "Magnitude (negligible/small/medium/large) is what matters at this N.")

Group          Median TTM(h)   Cliff delta vs Human        
------------------------------------------------------------
Human                   5.45
Copilot                12.87   delta=+0.140 (negligible) ***
Cursor                  0.83   delta=-0.327 (small) ***
Claude_Code             1.56   delta=-0.227 (small) ***
Devin                   1.94   delta=-0.178 (small) ***

BH-adjusted p; delta<0 => agent merges faster than Human. Magnitude (negligible/small/medium/large) is what matters at this N.


In [7]:
# Figure: time to merge — boxplot (capped at 48h for readability)
CAP = 48
plot_data = []
labels    = []
for agent in AGENTS:
    sub = agents_df[(agents_df['agent'] == agent) & agents_df['is_merged']]
    plot_data.append(sub['hours_to_merge'].clip(upper=CAP).dropna())
    labels.append(agent)
plot_data.append(h_ttm.clip(upper=CAP).dropna())
labels.append('Human')

fig, ax = plt.subplots(figsize=(9, 5))
bp = ax.boxplot(plot_data, labels=labels, patch_artist=True, notch=False)
for patch, label in zip(bp['boxes'], labels):
    patch.set_facecolor(AGENT_COLORS[label])
    patch.set_alpha(0.75)
ax.set_ylabel(f'Hours to Merge (capped at {CAP}h)')
ax.set_title('RQ6: Time to Merge Distribution per Agent vs Human')
fig.tight_layout()
save_fig(fig, 'rq6_time_to_merge', THEME3_DIR_V2)

  -> Saved: results\theme3_figures_v2\rq6_time_to_merge.png


WindowsPath('results/theme3_figures_v2/rq6_time_to_merge.png')

In [8]:
# Revision burden summary: % revised + median revision lines, with Cliff's delta on
# revision lines added (revised PRs only) vs Human. BH-adjusted across agents.
h_revised = human_rev[human_rev['num_commits'] > 1]
h_rev_pct = (human_rev['num_commits'] > 1).mean() * 100
h_rev_add = h_revised['rev_lines_added'].median()
h_rev_del = h_revised['rev_lines_deleted'].median()

pvals, rows = [], []
for agent in AGENTS:
    sub     = agent_rev[agent_rev['agent'] == agent]
    pct     = (sub['num_commits'] > 1).mean() * 100
    revised = sub[sub['num_commits'] > 1]
    _, p   = mann_whitney(revised['rev_lines_added'], h_revised['rev_lines_added'])
    d, mag = cliffs_delta(revised['rev_lines_added'], h_revised['rev_lines_added'])
    pvals.append(p)
    rows.append((agent, pct, revised['rev_lines_added'].median(),
                 revised['rev_lines_deleted'].median(), d, mag))
padj = bh_correct(pvals)

print("{:<13}{:>10}{:>14}{:>14}   {:<26}".format(
    'Group', '% Revised', 'Med RevLn+', 'Med RevLn-', 'Cliff delta(RevLn+) vs Human'))
print('-' * 80)
print("{:<13}{:>9.1f}%{:>14.0f}{:>14.0f}".format('Human', h_rev_pct, h_rev_add, h_rev_del))
for (agent, pct, add, dlt, d, mag), p in zip(rows, padj):
    print("{:<13}{:>9.1f}%{:>14.0f}{:>14.0f}   delta={:+.3f} ({}) {}".format(
        agent, pct, add, dlt, d, mag, sig_label(p)))
print("\nBH-adjusted p; delta>0 => agent's revisions add more lines than Human's.")

Group         % Revised    Med RevLn+    Med RevLn-   Cliff delta(RevLn+) vs Human
--------------------------------------------------------------------------------
Human             50.0%             6             4
Copilot           93.8%            30             9   delta=+0.260 (small) ***
Cursor            48.1%            10             6   delta=+0.082 (negligible) ***
Claude_Code       45.6%            12             8   delta=+0.077 (negligible) ***
Devin             56.4%             9             6   delta=+0.031 (negligible) ns

BH-adjusted p; delta>0 => agent's revisions add more lines than Human's.


In [9]:
# Figure: revision metrics bar chart (4 subplots)
metrics   = ['Merge Rate %', 'Median TTM (h)', '% Revised', 'Med Rev Lines+']
groups    = AGENTS + ['Human']
all_vals  = {g: [] for g in groups}

for g in groups:
    sub = agents_df[agents_df['agent'] == g] if g != 'Human' else human_df
    _, _, mr = merge_rate(sub)
    ttm = sub.loc[sub['is_merged'], 'hours_to_merge'].median()
    rev_sub = agent_rev[agent_rev['agent'] == g] if g != 'Human' else human_rev
    rev_pct = (rev_sub['num_commits'] > 1).mean() * 100
    rev_add = rev_sub.loc[rev_sub['num_commits'] > 1, 'rev_lines_added'].median()
    all_vals[g] = [mr, ttm, rev_pct, rev_add]

fig, axes = plt.subplots(1, 4, figsize=(16, 5))
for i, (ax, metric) in enumerate(zip(axes, metrics)):
    vals   = [all_vals[g][i] for g in groups]
    colors = [AGENT_COLORS[g] for g in groups]
    ax.bar(groups, vals, color=colors, edgecolor='white')
    ax.set_title(metric, fontsize=10)
    ax.set_xticklabels(groups, rotation=30, ha='right', fontsize=8)
fig.suptitle('RQ6: Agent Comparison — Key Bug-Fix Metrics', fontsize=12)
fig.tight_layout()
save_fig(fig, 'rq6_summary', THEME3_DIR_V2)

  -> Saved: results\theme3_figures_v2\rq6_summary.png


WindowsPath('results/theme3_figures_v2/rq6_summary.png')

## RQ8: AIDev period vs Post-AIDev period

Does agent bug-fixing performance change after the AIDev coverage ends?

- **AIDev period:** Dec 2024 – Jul 2025  
- **Post-AIDev period:** Aug 2025 – Feb 2026

In [10]:
# RQ8 uses the AIDev repo set ONLY, so a period difference reflects agent behaviour
# rather than a shift in which repos are present. Reload restricted to those repos.
df_rq8     = df            # already restricted to the AIDev repo set (v2)
agents_rq8 = agents_df.copy()

PERIODS = ['AIDev (Dec24-Jul25)', 'Post-AIDev (Aug25-Feb26)']
print('RQ8 validity check — PR counts per agent per period (AIDev repo set only):')
print("{:<14}{:>14}{:>16}".format('Agent', 'AIDev N', 'Post-AIDev N'))
print('-' * 46)
for agent in AGENTS:
    sub = agents_rq8[agents_rq8['agent'] == agent]
    n_ai   = int((sub['period'] == PERIODS[0]).sum())
    n_post = int((sub['period'] == PERIODS[1]).sum())
    flag = '  <-- thin, interpret with caution' if min(n_ai, n_post) < 100 else ''
    print("{:<14}{:>14,}{:>16,}{}".format(agent, n_ai, n_post, flag))

RQ8 validity check — PR counts per agent per period (AIDev repo set only):
Agent                AIDev N    Post-AIDev N
----------------------------------------------
Copilot                2,037           4,727
Cursor                10,391           7,976
Claude_Code            9,007           6,169
Devin                  1,270             707


In [11]:
# Merge rate per agent — AIDev vs Post-AIDev (AIDev repo set only)
rows_rq8 = []
for agent in AGENTS:
    sub = agents_rq8[agents_rq8['agent'] == agent]
    for period in PERIODS:
        p_sub = sub[sub['period'] == period]
        m, t, r = merge_rate(p_sub)
        rows_rq8.append({'Agent': agent, 'Period': period,
                         'Merged': m, 'Total': t, 'Rate': r})

rq8_df = pd.DataFrame(rows_rq8)
pivot  = rq8_df.pivot(index='Agent', columns='Period', values='Rate').round(1)
pivot.columns.name = None
pivot['Change (pp)'] = (
    pivot['Post-AIDev (Aug25-Feb26)'] - pivot['AIDev (Dec24-Jul25)']
).round(1)
print('Merge rate (%) — AIDev vs Post-AIDev (AIDev repo set only):')
print(pivot.to_string())

Merge rate (%) — AIDev vs Post-AIDev (AIDev repo set only):
             AIDev (Dec24-Jul25)  Post-AIDev (Aug25-Feb26)  Change (pp)
Agent                                                                  
Claude_Code                 84.8                      86.8          2.0
Copilot                     58.0                      64.2          6.2
Cursor                      90.8                      88.0         -2.8
Devin                       41.7                      55.7         14.0


In [12]:
# Is the period change significant per agent? OR (post vs AIDev) with 95% CI, BH-adjusted.
pvals, rows = [], []
for agent in AGENTS:
    sub  = agents_rq8[agents_rq8['agent'] == agent]
    ai   = sub[sub['period'] == 'AIDev (Dec24-Jul25)']
    post = sub[sub['period'] == 'Post-AIDev (Aug25-Feb26)']
    if len(ai) < MIN_N_PROP or len(post) < MIN_N_PROP:
        rows.append((agent, None)); pvals.append(1.0); continue
    ai_m, ai_t, ai_r = merge_rate(ai)
    po_m, po_t, po_r = merge_rate(post)
    _, p = chi_square(po_m, po_t, ai_m, ai_t)
    or_, lo, hi = odds_ratio_ci(po_m, po_t, ai_m, ai_t)   # post vs AIDev
    pvals.append(p)
    rows.append((agent, (ai_r, po_r, or_, lo, hi)))
padj = bh_correct(pvals)

print("{:<14}{:>10}{:>10}   {:<26}".format('Agent', 'AIDev%', 'Post%', 'OR post-vs-AIDev (95% CI)'))
print('-' * 64)
for (agent, vals), p in zip(rows, padj):
    if vals is None:
        print("{:<14}{:>30}".format(agent, 'insufficient N (< %d)' % MIN_N_PROP)); continue
    ai_r, po_r, or_, lo, hi = vals
    print("{:<14}{:>9.1f}%{:>9.1f}%   OR={:.2f} [{:.2f},{:.2f}] {}".format(
        agent, ai_r, po_r, or_, lo, hi, sig_label(p)))
print("\nBH-adjusted p; OR>1 => higher merge odds in the Post-AIDev period.")

Agent             AIDev%     Post%   OR post-vs-AIDev (95% CI) 
----------------------------------------------------------------
Copilot            58.0%     64.2%   OR=1.30 [1.17,1.44] ***
Cursor             90.8%     88.0%   OR=0.74 [0.68,0.82] ***
Claude_Code        84.8%     86.8%   OR=1.17 [1.07,1.29] ***
Devin              41.7%     55.7%   OR=1.76 [1.46,2.11] ***

BH-adjusted p; OR>1 => higher merge odds in the Post-AIDev period.


In [13]:
# Figure: grouped bar — AIDev vs Post-AIDev per agent (AIDev repo set only)
x     = list(range(len(AGENTS)))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
for i, period in enumerate(PERIODS):
    vals   = [rq8_df[(rq8_df['Agent'] == a) & (rq8_df['Period'] == period)]['Rate'].values[0]
              for a in AGENTS]
    offset = (i - 0.5) * width
    bars   = ax.bar([xi + offset for xi in x], vals, width,
                    label=period, alpha=0.85)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                f'{v:.1f}%', ha='center', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(AGENTS)
ax.set_ylabel('Merge Rate (%)')
ax.set_ylim(0, 105)
ax.set_title('RQ8: Bug-Fix Acceptance Rate — AIDev vs Post-AIDev (AIDev repo set only)')
ax.legend()
fig.tight_layout()
save_fig(fig, 'rq8_aidev_vs_post_aidev', THEME3_DIR_V2)

  -> Saved: results\theme3_figures_v2\rq8_aidev_vs_post_aidev.png


WindowsPath('results/theme3_figures_v2/rq8_aidev_vs_post_aidev.png')

In [14]:
# Time to merge: AIDev vs Post-AIDev per agent (AIDev repo set), Cliff's delta + BH.
merged_rq8 = agents_rq8[agents_rq8['is_merged']]
pvals, rows = [], []
for agent in AGENTS:
    sub  = merged_rq8[merged_rq8['agent'] == agent]
    ai   = sub[sub['period'] == 'AIDev (Dec24-Jul25)']['hours_to_merge']
    post = sub[sub['period'] == 'Post-AIDev (Aug25-Feb26)']['hours_to_merge']
    if len(ai) < MIN_N_MEDIAN or len(post) < MIN_N_MEDIAN:
        rows.append((agent, None)); pvals.append(1.0); continue
    _, p   = mann_whitney(ai, post)
    d, mag = cliffs_delta(post, ai)   # delta>0 => slower (more hours) post-AIDev
    pvals.append(p)
    rows.append((agent, (ai.median(), post.median(), d, mag)))
padj = bh_correct(pvals)

print("{:<14}{:>14}{:>14}   {:<26}".format(
    'Agent', 'AIDev TTM(h)', 'Post TTM(h)', "Cliff delta (post vs AIDev)"))
print('-' * 70)
for (agent, vals), p in zip(rows, padj):
    if vals is None:
        print("{:<14}{:>30}".format(agent, 'insufficient N')); continue
    ai_med, po_med, d, mag = vals
    print("{:<14}{:>14.2f}{:>14.2f}   delta={:+.3f} ({}) {}".format(
        agent, ai_med, po_med, d, mag, sig_label(p)))
print("\nBH-adjusted p; delta>0 => slower merges in Post-AIDev period.")

Agent           AIDev TTM(h)   Post TTM(h)   Cliff delta (post vs AIDev)
----------------------------------------------------------------------
Copilot                19.17         11.14   delta=-0.054 (negligible) **
Cursor                  0.76          0.91   delta=+0.042 (negligible) ***
Claude_Code             1.59          1.53   delta=+0.009 (negligible) ns
Devin                   1.45          2.50   delta=+0.117 (negligible) **

BH-adjusted p; delta>0 => slower merges in Post-AIDev period.


## Methodology: `type == 'fix'` label validation

Every RQ rests on the `type == 'fix'` classification, which has unknown precision/recall that may differ by agent. The cell below exports a stratified sample (50 PRs per agent + 50 human) to `results/label_validation_sample.csv` for **manual labelling**. After hand-labelling a `true_is_fix` column, re-load it to compute per-agent precision and re-run the headline figures on the high-confidence subset as a sensitivity check.

In [15]:
# Export a stratified sample for manual label validation (50 per agent + 50 human).
SAMPLE_PER_GROUP = 50
sample_cols = ['id', 'number', 'repo', 'agent', 'source', 'title', 'html_url']
parts = []
for agent in AGENTS:
    sub = agents_df[agents_df['agent'] == agent]
    parts.append(sub.sample(min(SAMPLE_PER_GROUP, len(sub)), random_state=0))
parts.append(human_df.sample(min(SAMPLE_PER_GROUP, len(human_df)), random_state=0))

label_sample = pd.concat(parts)[sample_cols].copy()
label_sample['true_is_fix'] = ''   # <- fill manually: 1 = genuine bug fix, 0 = not
out_path = THEME3_DIR_V2.parent / 'label_validation_sample.csv'
label_sample.to_csv(out_path, index=False)
print(f"Wrote {len(label_sample)} rows to {out_path}")
print("Fill the 'true_is_fix' column by hand, then compute precision per agent.")

Wrote 250 rows to results\label_validation_sample.csv
Fill the 'true_is_fix' column by hand, then compute precision per agent.


## Threats to validity (Theme 3)

- **"Best agent" depends on the metric and the baseline.** Rankings can flip between merge rate, TTM, and revision burden, and between the unmatched and matched human baselines. Report the matched-baseline odds ratios as primary.
- **Significance ≠ importance.** At these sample sizes nearly everything is statistically significant; we report odds ratios / Cliff's delta and treat OR≈1 or negligible delta as no practical difference regardless of stars.
- **Model-version drift.** Each agent spans multiple underlying models over the window, so a single per-agent number averages over several products. RQ8's period split partly captures this but cannot isolate it.
- **RQ8 thinness.** Restricting to the AIDev repo set shrinks per-agent, per-period samples (see the validity check above); agents flagged "thin" should not carry a period conclusion. Devin is sparse throughout (`investigate.py`).
- **Auto-merge & review rounds unobservable.** As in Theme 2, merge rate and TTM partly reflect repo merge policy, and revision burden omits review-comment round-trips — neither is recoverable from the available columns.
- **Label dependence.** All results assume the `type == 'fix'` classifier is accurate; until the validation sample above is hand-labelled, per-agent label-precision differences remain an unquantified confound.
- **First-commit ordering.** Revision-line attribution relies on parquet row order to pick the first commit (no commit timestamp exists), which may misattribute revision effort.